# Oak Forest, IL — LVT Policy Analysis

Models a revenue-neutral 4:1 split-rate LVT shift for parcels inside the
**City of Oak Forest** (Bremen Township, Cook County, IL).

## Policy decisions

| Choice | Setting | Note |
|---|---|---|
| Geographic scope | City of Oak Forest (all 10,701 parcels in CCAO universe) | Parcel Universe pre-filtered to municipality |
| Levy scope | City of Oak Forest only | Agency `030900000`; TY2023 levy = $11,860,299 |
| Reform shape | Split-rate at 4:1 | Land taxed at 4× the improvement millage; revenue-neutral |
| Exemptions | Preserve all | Cook County exemptions (Homestead, Senior, etc.) already reflected in PTAXSIM |
| TIF districts | Transparent | PTAXSIM city_tax reflects base-EAV share; 453 parcels across 5 TIF districts |
| Base year | Tax Year 2023 (payable 2024) | PTAXSIM 2024 database; CCAO board-certified AV |

**Data sources (all from sibling project `oak-forest-profit-loss-map`):**
- **CCAO Parcel Universe** — 10,701 Oak Forest PINs with class codes, centroids, TIF flags, Census block group GEOIDs
- **CCAO Historic Assessed Values** — board-certified land AV and building AV per PIN for TY2023
- **PTAXSIM 2024** (`ptaxsim-2024.0.0.db`) — per-PIN city levy via proportional rate method; handles TIF base-EAV correctly

**Cook County assessment context:**
- Assessed values are fractional: Class 2 residential = 10% of market value; Class 5/6 commercial/industrial = 25%
- EAV = AV × Cook County equalization factor (3.0163 for TY2023)
- Land and building assessed separately; taxable EAV = (AV − exemptions) × EQ factor
- PTAXSIM gap: proportional method yields ~2% above official levy; scaled to match exactly

**Improvement ratio caveat:**
- CCAO's assessments routinely assign default IR ratios rather than parcel-specific land/building splits.
  The distribution of improvement ratios should be checked for piling at uniform values.
  This does not affect total revenue neutrality but does affect individual parcel outcomes.

## Section 1 — Imports and configuration

In [1]:
import sys
import os
import sqlite3
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from dotenv import load_dotenv

sys.path.insert(0, '../..')
REPO_ROOT = Path('../..').resolve()
load_dotenv(REPO_ROOT / '.env')

from lvt.lvt_utils import model_split_rate_tax, save_standard_export, calculate_category_tax_summary

# ── Configuration ────────────────────────────────────────────────────────────
CITY_NAME              = 'oak_forest'
STATE_FIPS             = '17'           # Illinois
COUNTY_FIPS            = '031'          # Cook County
TAX_YEAR               = 2023
EQ_FACTOR              = 3.0163         # Cook County TY2023 final equalization factor
CITY_LEVY              = 11_860_299.0   # Official TY2023 city levy (PTAXSIM agency table)
CITY_RATE              = 1.831          # Per $1,000 EAV; uniform across all Oak Forest tax codes
LAND_IMPROVEMENT_RATIO = 4.0
MODEL_TYPE             = 'split_rate:4.0'
CITY_AGENCY            = '030900000'    # CITY OF OAK FOREST

# Sibling project data paths
SIBLING    = Path('C:/projects/oak-forest-profit-loss-map')
PTAXSIM_DB = SIBLING / 'data/ptaxsim/ptaxsim-2024.0.0.db'
CCAO_DIR   = SIBLING / 'data/ccao'

print(f'PTAXSIM DB exists: {PTAXSIM_DB.exists()}')
print(f'CCAO dir exists:   {CCAO_DIR.exists()}')

PTAXSIM DB exists: True
CCAO dir exists:   True


## Section 2 — Load CCAO parcel data

In [2]:
print('Loading CCAO Parcel Universe...')
universe = pd.read_parquet(CCAO_DIR / 'parcel_universe.parquet')
universe['pin'] = universe['pin'].astype(str).str.strip().str.zfill(14)
print(f'  {len(universe):,} Oak Forest parcels')

print('\nLoading CCAO Historic Assessed Values (TY2023)...')
av_raw = pd.read_parquet(CCAO_DIR / 'historic_av.parquet')
av_raw['pin'] = av_raw['pin'].astype(str).str.strip().str.zfill(14)
for col in ['board_land', 'board_bldg', 'board_tot']:
    av_raw[col] = pd.to_numeric(av_raw[col], errors='coerce').fillna(0.0)

# Use board-certified values (post-Board of Review); deduplicate to one row per PIN
av = (
    av_raw.sort_values('board_tot', ascending=False)
    .drop_duplicates(subset='pin', keep='first')
    [['pin', 'board_land', 'board_bldg', 'board_tot']]
)
print(f'  {len(av):,} AV records after deduplication')
print(f'  Parcels with board_land > 0: {(av["board_land"] > 0).sum():,}')
print(f'  Parcels with board_bldg > 0: {(av["board_bldg"] > 0).sum():,}')

Loading CCAO Parcel Universe...
  10,701 Oak Forest parcels

Loading CCAO Historic Assessed Values (TY2023)...
  52,693 AV records after deduplication
  Parcels with board_land > 0: 47,464
  Parcels with board_bldg > 0: 42,427


In [3]:
print('Querying PTAXSIM for per-PIN city levy (TY2023)...')

# Filter to Oak Forest tax codes first (53 codes) to avoid scanning 1.86M Cook County pins.
PTAXSIM_QUERY = '''
    WITH city_tcs AS (
        SELECT DISTINCT tax_code_num FROM tax_code
        WHERE year = ? AND agency_num = ?
    ),
    city_rates AS (
        SELECT year, tax_code_num, agency_rate AS city_rate
        FROM tax_code WHERE year = ? AND agency_num = ?
    ),
    total_rates AS (
        SELECT year, tax_code_num, SUM(agency_rate) AS total_rate
        FROM tax_code
        WHERE year = ? AND tax_code_num IN (SELECT tax_code_num FROM city_tcs)
        GROUP BY year, tax_code_num
    )
    SELECT
        p.pin, p.av_clerk,
        p.exe_homeowner, p.exe_senior, p.exe_freeze,
        p.exe_longtime_homeowner, p.exe_disabled,
        p.exe_vet_returning, p.exe_vet_dis_lt50, p.exe_vet_dis_50_69,
        p.exe_vet_dis_ge70, p.exe_vet_dis_100, p.exe_wwii, p.exe_abate,
        CASE WHEN tr.total_rate > 0
             THEN p.tax_bill_total * cr.city_rate / tr.total_rate
             ELSE 0.0 END AS city_tax_raw
    FROM pin p
    JOIN city_rates cr ON p.year = cr.year AND p.tax_code_num = cr.tax_code_num
    JOIN total_rates tr ON p.year = tr.year AND p.tax_code_num = tr.tax_code_num
    WHERE p.year = ?
      AND p.tax_code_num IN (SELECT tax_code_num FROM city_tcs)
'''

with sqlite3.connect(PTAXSIM_DB) as conn:
    ptax = pd.read_sql_query(
        PTAXSIM_QUERY, conn,
        params=[TAX_YEAR, CITY_AGENCY,
                TAX_YEAR, CITY_AGENCY,
                TAX_YEAR,
                TAX_YEAR]
    )

ptax['pin'] = ptax['pin'].astype(str).str.zfill(14)

EXE_COLS = [
    'exe_homeowner', 'exe_senior', 'exe_freeze', 'exe_longtime_homeowner',
    'exe_disabled', 'exe_vet_returning', 'exe_vet_dis_lt50', 'exe_vet_dis_50_69',
    'exe_vet_dis_ge70', 'exe_vet_dis_100', 'exe_wwii', 'exe_abate',
]
ptax['total_exe'] = ptax[EXE_COLS].sum(axis=1)

# Scale to match official levy exactly (proportional method ~2% high)
raw_total = ptax['city_tax_raw'].sum()
scale = CITY_LEVY / raw_total
ptax['city_tax'] = ptax['city_tax_raw'] * scale

print(f'  PTAXSIM raw total: ${raw_total:,.0f}')
print(f'  Scale factor:      {scale:.6f}')
print(f'  Scaled total:      ${ptax["city_tax"].sum():,.0f}  (target: ${CITY_LEVY:,.0f})')
print(f'  PINs returned:     {len(ptax):,}')


Querying PTAXSIM for per-PIN city levy (TY2023)...


  PTAXSIM raw total: $12,109,555
  Scale factor:      0.979417
  Scaled total:      $11,860,299  (target: $11,860,299)
  PINs returned:     10,699


## Section 3 — Merge, classify, and compute taxable base

In [4]:
print('Merging parcel universe + AV + PTAXSIM...')

df = universe.copy()

df = df.merge(av, on='pin', how='left')
df['board_land'] = df['board_land'].fillna(0.0)
df['board_bldg'] = df['board_bldg'].fillna(0.0)
df['board_tot']  = df['board_tot'].fillna(0.0)

df = df.merge(
    ptax[['pin', 'city_tax', 'av_clerk', 'total_exe']],
    on='pin', how='left'
)
df['city_tax']   = df['city_tax'].fillna(0.0)
df['av_clerk']   = df['av_clerk'].fillna(0.0)
df['total_exe']  = df['total_exe'].fillna(0.0)

# TIF flag
df['in_tif'] = (
    df['tax_tif_district_name'].notna() &
    (df['tax_tif_district_name'].astype(str).str.strip() != '')
)

print(f'  DataFrame: {len(df):,} rows')
print(f'  city_tax sum: ${df["city_tax"].sum():,.0f}')
print(f'  Parcels in TIF districts: {df["in_tif"].sum():,}')
print()
print('TIF district breakdown:')
print(
    df[df['in_tif']]['tax_tif_district_name']
    .value_counts().to_string()
)

Merging parcel universe + AV + PTAXSIM...
  DataFrame: 10,701 rows
  city_tax sum: $11,854,890
  Parcels in TIF districts: 453

TIF district breakdown:
tax_tif_district_name
TIF - OAK FOREST - CICERO AVE                 151
TIF - OAK FOREST - 3 159TH ST / CICERO AVE    137
TIF - OAK FOREST - 7                          107
TIF - OAK FOREST - BUSINESS PARK EAST          53
TIF - OAK FOREST - 4                            5


In [5]:
# ── Improvement ratio from board-certified AV ──────────────────────────────
# IR = board_bldg / board_tot; 0 for vacant/exempt (board_tot == 0)
board_tot_pos = df['board_tot'].replace(0, np.nan)
df['improvement_ratio'] = (df['board_bldg'] / board_tot_pos).fillna(0.0).clip(0, 1)

# ── Taxable EAV (effective, per-parcel, city-levy base) ───────────────────
# Back-calculate from city_tax so the reform model uses the exact same base
# the city currently taxes (handles TIF base-EAV constraint automatically).
df['effective_taxable_eav'] = (df['city_tax'] / (CITY_RATE / 1000.0)).clip(lower=0.0)

df['taxable_land_eav'] = df['effective_taxable_eav'] * (1.0 - df['improvement_ratio'])
df['taxable_bldg_eav'] = df['effective_taxable_eav'] * df['improvement_ratio']

print('Taxable EAV summary:')
total_eav = df['effective_taxable_eav'].sum()
land_eav  = df['taxable_land_eav'].sum()
bldg_eav  = df['taxable_bldg_eav'].sum()
print(f'  Total effective taxable EAV: ${total_eav:,.0f}')
print(f'  Land EAV:  ${land_eav:,.0f}  ({land_eav/total_eav*100:.1f}%)')
print(f'  Bldg EAV:  ${bldg_eav:,.0f}  ({bldg_eav/total_eav*100:.1f}%)')

# Improvement ratio distribution — check for CCAO default piling
ir_nonzero = df.loc[df['improvement_ratio'] > 0, 'improvement_ratio']
print(f'\nImprovement ratio (non-zero parcels, n={len(ir_nonzero):,}):')
print(f'  Median: {ir_nonzero.median():.3f}')
print(f'  Mean:   {ir_nonzero.mean():.3f}')
print(f'  % at exactly 0.500: {(ir_nonzero.round(3) == 0.500).mean()*100:.1f}%')
print(f'  % at exactly 0.800: {(ir_nonzero.round(3) == 0.800).mean()*100:.1f}%')
print(f'  % at exactly 0.900: {(ir_nonzero.round(3) == 0.900).mean()*100:.1f}%')

Taxable EAV summary:
  Total effective taxable EAV: $6,474,543,797
  Land EAV:  $1,316,422,902  (20.3%)
  Bldg EAV:  $5,158,120,895  (79.7%)

Improvement ratio (non-zero parcels, n=10,103):
  Median: 0.852
  Mean:   0.825
  % at exactly 0.500: 0.1%
  % at exactly 0.800: 0.2%
  % at exactly 0.900: 0.1%


In [6]:
# ── Property category classification ──────────────────────────────────────
# Cook County major class codes:
#   0xx / EX = government/exempt   4xx = not-for-profit/charitable
#   1xx = vacant land              5xx = commercial
#   2xx = residential (SFR, flat)  6xx = industrial
#   3xx = multi-family apartment   8xx = railroad/misc
#   9xx = condominium

def classify_parcel(cls_code: str) -> str:
    cls = str(cls_code).strip().upper()
    if not cls:
        return 'Other'
    major = cls[0]
    if major in ('E', 'X') or major == '0' or major == '4':
        return 'Exempt'
    if major == '1':
        return 'Vacant Land'
    if major == '2':
        if cls in ('211', '212', '213', '214'):
            return 'Small Multi-Family (2-4 units)'
        if cls == '215':
            return 'Large Multi-Family (5+ units)'
        return 'Single Family Residential'
    if major == '3':
        return 'Large Multi-Family (5+ units)'
    if major == '5':
        return 'Commercial'
    if major == '6':
        return 'Industrial'
    if major == '9':
        return 'Condominium'
    if major == '8':
        return 'Other'
    return 'Other'

df['PROPERTY_CATEGORY'] = df['class'].apply(classify_parcel)

print('Property category distribution:')
cat_counts = df['PROPERTY_CATEGORY'].value_counts()
print(cat_counts.to_string())
print(f'\nTotal: {cat_counts.sum():,}')

Property category distribution:
PROPERTY_CATEGORY
Single Family Residential         9654
Commercial                         353
Exempt                             240
Vacant Land                        225
Small Multi-Family (2-4 units)     124
Other                               47
Large Multi-Family (5+ units)       45
Industrial                          11
Condominium                          2

Total: 10,701


## Section 4 — Validate current tax

In [7]:
# Validate: sum of city_tax vs official levy
# Note: city_tax will be renamed to current_tax before modeling
total_current = df["city_tax"].sum()
print(f"Current tax total:  ${total_current:,.0f}")
print(f"Official levy:      ${CITY_LEVY:,.0f}")
print(f"Gap:                {(total_current - CITY_LEVY)/CITY_LEVY*100:.3f}%")
print()
print("By property category:")
cat_preview = (
    df.groupby("PROPERTY_CATEGORY")
    .agg(
        parcels=("city_tax", "count"),
        current_tax=("city_tax", "sum"),
        taxable_eav=("effective_taxable_eav", "sum"),
        avg_IR=("improvement_ratio", "mean"),
        in_tif=("in_tif", "sum"),
    )
    .sort_values("current_tax", ascending=False)
)
cat_preview["pct_levy"] = cat_preview["current_tax"] / cat_preview["current_tax"].sum() * 100
print(cat_preview.round(1).to_string())
print()
exempt_tax = df.loc[df["PROPERTY_CATEGORY"] == "Exempt", "city_tax"].sum()
print(f"Exempt parcels city_tax: ${exempt_tax:,.2f}  (should be ~0)")

Current tax total:  $11,854,890
Official levy:      $11,860,299
Gap:                -0.046%

By property category:
                                parcels  current_tax   taxable_eav  avg_IR  in_tif  pct_levy
PROPERTY_CATEGORY                                                                           
Single Family Residential          9654    9463976.2  5.168747e+09     0.8     137      79.8
Commercial                          353    1528596.7  8.348425e+08     0.5     154      12.9
Small Multi-Family (2-4 units)      124     335730.7  1.833592e+08     0.9      15       2.8
Large Multi-Family (5+ units)        45     230299.4  1.257779e+08     0.8      26       1.9
Other                                47     164401.9  8.978805e+07     0.6      26       1.4
Vacant Land                         225      70038.1  3.825127e+07     0.0      30       0.6
Industrial                           11      31492.1  1.719938e+07     0.5       9       0.3
Exempt                              240      200

## Section 5 — Model 4:1 split-rate LVT

In [8]:
# Rename city_tax → current_tax before modeling (required by model_split_rate_tax)
df = df.rename(columns={'city_tax': 'current_tax'})

# Exclude fully-exempt parcels from the revenue-neutral solve
# (they have no taxable EAV and would distort the millage calculation)
exempt_mask = df['PROPERTY_CATEGORY'] == 'Exempt'

land_millage, improvement_millage, total_revenue, df = model_split_rate_tax(
    df,
    land_value_col='taxable_land_eav',
    improvement_value_col='taxable_bldg_eav',
    current_revenue=CITY_LEVY,
    land_improvement_ratio=LAND_IMPROVEMENT_RATIO,
    exclude_mask=exempt_mask,
    verbose=True,
)

print(f'\nMillage comparison (per $1,000 EAV):')
print(f'  Current uniform:   {CITY_RATE:.4f}')
print(f'  New land:          {land_millage:.4f}  (+{(land_millage/CITY_RATE - 1)*100:.1f}%)')
print(f'  New improvement:   {improvement_millage:.4f}  ({(improvement_millage/CITY_RATE - 1)*100:.1f}%)')
print(f'  Land/imp ratio:    {land_millage/improvement_millage:.2f}:1  (target: 4.0:1)')
print(f'  Modeled revenue:   ${total_revenue:,.0f}')
print(f'  Revenue gap:       {(total_revenue - CITY_LEVY)/CITY_LEVY*100:.4f}%')

Split-rate tax model (Land:Improvement = 4.0:1)
Land millage rate: 4.5493
Improvement millage rate: 1.1373
Total tax revenue: $11,840,246.35
Target revenue: $11,840,246.35
Revenue difference: $0.00 (0.0000%)

Millage comparison (per $1,000 EAV):
  Current uniform:   1.8310
  New land:          4.5493  (+148.5%)
  New improvement:   1.1373  (-37.9%)
  Land/imp ratio:    4.00:1  (target: 4.0:1)
  Modeled revenue:   $11,860,299
  Revenue gap:       0.0000%


In [9]:
cat_summary = calculate_category_tax_summary(
    df,
    category_col="PROPERTY_CATEGORY",
    current_tax_col="current_tax",
    new_tax_col="new_tax",
)
print("Category summary:")
print(cat_summary.to_string())
print()
print("TIF vs non-TIF (taxable parcels only):")
taxable = df[df["PROPERTY_CATEGORY"] != "Exempt"]
tif_summary = (
    taxable.groupby("in_tif")
    .agg(
        parcels=("current_tax", "count"),
        current_tax=("current_tax", "sum"),
        new_tax=("new_tax", "sum"),
        land_eav=("taxable_land_eav", "sum"),
        bldg_eav=("taxable_bldg_eav", "sum"),
    )
)
tif_summary["pct_change"] = (tif_summary["new_tax"] - tif_summary["current_tax"]) / tif_summary["current_tax"] * 100
print(tif_summary.round(1).to_string())

Category summary:
                PROPERTY_CATEGORY  total_tax_change_dollars  property_count  mean_tax_change  median_tax_change  mean_tax_change_pct  median_tax_change_pct  total_current_tax  total_new_tax  pct_increase_gt_threshold  pct_decrease_gt_threshold  total_tax_change_pct
6       Single Family Residential            -597908.899627            9654       -61.933799         -88.040492            -5.482410             -10.212174       9.463976e+06   8.866067e+06                   8.980733                  51.408742             -6.317735
0                      Commercial             578525.507863             353      1638.882459         824.482866            58.823332              45.742980       1.528597e+06   2.107122e+06                  85.552408                   1.699717             37.846838
2                          Exempt                  0.000000             240         0.000000           0.000000             0.000000               0.000000       2.005265e+04   2.00526

## Section 6 — Visualizations

In [11]:
# Tax change scatter: estimated land market value vs pct change (SFR only)
sfr = df[df["PROPERTY_CATEGORY"] == "Single Family Residential"].copy()
sfr_taxable = sfr[sfr["current_tax"] > 0].copy()
sfr_taxable["land_market_value"] = sfr_taxable["board_land"] * EQ_FACTOR / 0.10

fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(
    sfr_taxable["land_market_value"].clip(upper=sfr_taxable["land_market_value"].quantile(0.98)),
    sfr_taxable["tax_change_pct"].clip(-80, 80),
    alpha=0.3, s=8, c="steelblue"
)
ax.axhline(0, color="black", linewidth=0.8, linestyle="--")
ax.set_xlabel("Estimated Land Market Value ($)")
ax.set_ylabel("Tax Change (%)")
ax.set_title("Oak Forest SFR - Tax Change vs Land Value (4:1 Split-Rate)")
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"K"))
plt.tight_layout()
plt.savefig("../../analysis/reports/oak_forest/sfr_scatter.png", dpi=150)
plt.show()
n_sfr = len(sfr_taxable)
n_down = (sfr_taxable["tax_change"] < 0).sum()
n_up = (sfr_taxable["tax_change"] > 0).sum()
print(f"SFR parcels: {n_sfr:,}")
print(f"Median tax change: {sfr_taxable["tax_change_pct"].median():.1f}%")
print(f"Tax decreases: {n_down:,} ({n_down/n_sfr*100:.1f}%)")
print(f"Tax increases: {n_up:,} ({n_up/n_sfr*100:.1f}%)")

SFR parcels: 9,525
Median tax change: -10.3%
Tax decreases: 8,031 (84.3%)
Tax increases: 1,494 (15.7%)


C:\Users\druss\AppData\Local\Temp\claude\ipykernel_104808\2965478271.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Section 7 — Census equity analysis

In [13]:
# CCAO Parcel Universe already has census_block_group_geoid pre-computed.
# Use it directly as std_geoid; fetch ACS demographics via get_census_data
# (ACS-only, no TIGERweb boundaries — much faster than get_census_data_with_boundaries).
df['std_geoid'] = df['census_block_group_geoid'].astype(str).str.strip()
print('std_geoid coverage:', df['std_geoid'].notna().mean())
print('Unique block groups:', df['std_geoid'].nunique())

from lvt.census_utils import get_census_data
census_data = None
try:
    census_data = get_census_data(STATE_FIPS + COUNTY_FIPS, year=2022)
    print('Census ACS fetched:', len(census_data), 'block groups (Cook County)')
    oak_bgs = set(df['std_geoid'].dropna().unique())
    census_data = census_data[census_data['std_geoid'].isin(oak_bgs)]
    print('After filter:', len(census_data), 'Oak Forest block groups matched')
except Exception as e:
    print('Census fetch failed:', e)
    census_data = None


std_geoid coverage: 1.0
Unique block groups: 24


Census ACS fetched: 4002 block groups (Cook County)
After filter: 24 Oak Forest block groups matched


In [14]:
_demo_cols = ['std_geoid', 'median_income', 'minority_pct', 'black_pct', 'total_pop']
if census_data is not None:
    _available = [c for c in _demo_cols if c in census_data.columns]
    df = df.merge(census_data[_available], on='std_geoid', how='left')
    matched = df['median_income'].notna().mean() if 'median_income' in df.columns else 0
    print('Census merge:', round(matched * 100, 1), '% of parcels matched')
    if 'median_income' in df.columns:
        inc = df['median_income'].dropna()
        print('Median income range:', inc.min(), 'to', inc.max())
else:
    for _col in ['median_income', 'minority_pct', 'black_pct']:
        df[_col] = float('nan')
    print('Demographics set to NaN (census fetch failed)')


Census merge: 100.0 % of parcels matched
Median income range: 36667.0 to 154926.0


In [15]:
# Equity analysis: tax change by income quintile (residential only)
if 'median_income' in df.columns and df['median_income'].notna().any():
    res_df = df[
        df['PROPERTY_CATEGORY'].isin([
            'Single Family Residential', 'Small Multi-Family (2-4 units)',
            'Large Multi-Family (5+ units)', 'Condominium'
        ]) & df['current_tax'].gt(0) & df['median_income'].notna()
    ].copy()

    res_df['income_quintile'] = pd.qcut(
        res_df['median_income'], 5,
        labels=['Q1 (lowest)', 'Q2', 'Q3', 'Q4', 'Q5 (highest)']
    )

    eq_summary = (
        res_df.groupby('income_quintile')
        .agg(
            parcels=('current_tax', 'count'),
            med_income=('median_income', 'median'),
            current_tax=('current_tax', 'sum'),
            new_tax=('new_tax', 'sum'),
            med_pct_change=('tax_change_pct', 'median'),
        )
    )
    eq_summary['pct_change_total'] = (
        (eq_summary['new_tax'] - eq_summary['current_tax']) / eq_summary['current_tax'] * 100
    )
    print('Tax change by income quintile (residential parcels):')
    print(eq_summary.round(1).to_string())
else:
    print('Skipping equity analysis — census data not available')

Tax change by income quintile (residential parcels):
                 parcels  med_income  current_tax    new_tax  med_pct_change  pct_change_total
income_quintile                                                                               
Q1 (lowest)         2475     64861.0    2043408.9  1729488.4           -16.9             -15.4
Q2                  1516     88182.0    1559775.9  1484178.7            -9.7              -4.8
Q3                  2098     94531.0    2026798.9  1882446.8            -9.8              -7.1
Q4                  2290    111607.0    2709441.0  2616093.8            -7.9              -3.4
Q5 (highest)        1317    142778.0    1700883.5  1612918.6            -8.8              -5.2


## Section 8 — Export

In [16]:
import os
os.makedirs('../../analysis/data', exist_ok=True)

out_df = save_standard_export(
    df=df,
    city=CITY_NAME,
    output_path=f'../../analysis/data/{CITY_NAME}.csv',
    model_type=MODEL_TYPE,
    land_millage=land_millage,
    improvement_millage=improvement_millage,
    property_category_col='PROPERTY_CATEGORY',
    current_tax_col='current_tax',
    new_tax_col='new_tax',
    tax_change_col='tax_change',
    tax_change_pct_col='tax_change_pct',
    taxable_land_col='taxable_land_value',
    taxable_improvement_col='taxable_improvement_value',
)

print(f'Exported {len(out_df):,} rows to analysis/data/{CITY_NAME}.csv')
print(f'Revenue check: ${out_df["current_tax"].sum():,.0f} current / ${out_df["new_tax"].sum():,.0f} modeled')
print(f'Columns: {list(out_df.columns)}')

  ✓ oak_forest: 10,701 rows → ../../analysis/data/oak_forest.csv  [model: split_rate:4.0]


Exported 10,701 rows to analysis/data/oak_forest.csv
Revenue check: $11,854,890 current / $11,860,299 modeled
Columns: ['city', 'property_category', 'current_tax', 'new_tax', 'tax_change', 'tax_change_pct', 'taxable_land_value', 'taxable_improvement_value', 'is_fully_exempt', 'std_geoid', 'median_income', 'minority_pct', 'black_pct', 'model_type', 'land_millage', 'improvement_millage']


In [17]:
from lvt.viz import create_city_report
create_city_report(out_df, CITY_NAME, show=False)
print('Done — standard report PNGs written to analysis/reports/oak_forest/')

Done — standard report PNGs written to analysis/reports/oak_forest/
